In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain 
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

C:\Users\Vibhav Kaushik V\AppData\Local\Temp\ipykernel_35064\2990640704.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
import sys
print(sys.executable)

d:\Sub Volume- D\Rag Agent\.venv\Scripts\python.exe


In [ ]:
os.environ["GOOGLE_API_KEY"] = ""

In [6]:
#Load the dataset
loader = TextLoader("syllabus.txt")
docs = loader.load()

In [7]:
# 2. Split Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap = 50
)
chunks = text_splitter.split_documents(docs)

In [12]:
# 3. Embed and store ( Using google embedding model)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = FAISS.from_documents(chunks,embeddings)

In [13]:
# 4. Set up Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [19]:
# 5. Create the Chain (Prompt + Gemini LLM + Retrieval)
# Using gemini-1.5-flash as it is fast and heavily supported on the free tier
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [21]:
system_prompt = (
    "You are a helpful teaching assistant. Use the retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know."
    "\n\nContext: {context}"
)

In [22]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [23]:
# Assemble the RAG pipeline
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [24]:
# 6. Execute
response = rag_chain.invoke({"input": "What are the grading criteria for this course?"})
print(response["answer"])

The grading criteria for this course are:
*   **Homework Assignments:** 30%
*   **Midterm Exam:** 30%
*   **Final Project:** 40%
